# Статистический анализ всех полей Сферы и ЕГРН

Ноутбук сам подключается к Сфере и КХД. Он собирает расширенный слой всех объектов недвижимости, добавляет все поля таблиц Сферы, участвующих в объектной цепочке, разворачивает `characteristics JSONB`, затем добавляет все колонки однозначно найденной записи `EGRN_DATA`.

Промежуточный CSV с объектами не создаётся. Сохраняется только один агрегированный паспорт признаков. Реальные адреса, ИНН, названия компаний, номера договоров и кадастровые номера в частотные значения паспорта не выводятся.

Таблицы Сферы, не связанные с текущей объектной выборкой, в анализ не включаются: они не являются признаками одной строки объекта.


In [1]:
%pip install pandas numpy sqlalchemy "psycopg[binary]" oracledb openpyxl


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
from pathlib import Path
import oracledb
import numpy as np
import pandas as pd
import re
from sqlalchemy import URL, create_engine, text

pd.set_option('display.max_columns', 100)

In [3]:
CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / 'уч данные.txt').exists():
    NOTEBOOK_DIR = CURRENT_DIR
elif (CURRENT_DIR / 'notebooks' / 'уч данные.txt').exists():
    NOTEBOOK_DIR = CURRENT_DIR / 'notebooks'
else:
    NOTEBOOK_DIR = CURRENT_DIR

PROJECT_ROOT = (
    NOTEBOOK_DIR.parent
    if NOTEBOOK_DIR.name == 'notebooks'
    else NOTEBOOK_DIR
)
OUTPUT_DIR = PROJECT_ROOT / 'РЕЗУЛЬТАТЫ_ЛОКАЛЬНО'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Корень проекта:', PROJECT_ROOT)
print('Папка результатов:', OUTPUT_DIR)

Корень проекта: t:\Блок актуарных расчетов\Управление актуарных расчетов\Общая\Светова\риск моделирование\анализ таблиц\sql python
Папка результатов: t:\Блок актуарных расчетов\Управление актуарных расчетов\Общая\Светова\риск моделирование\анализ таблиц\sql python\РЕЗУЛЬТАТЫ_ЛОКАЛЬНО


# 1. Подключение к Сфере


In [4]:
CREDENTIALS_PATH = NOTEBOOK_DIR / 'уч данные.txt'

def read_credentials(path):
    if not path.exists():
        raise FileNotFoundError(f'Не найден файл с учётными данными: {path}')

    credentials = {}
    for line_number, raw_line in enumerate(
        path.read_text(encoding='utf-8-sig').splitlines(),
        start=1,
    ):
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        if '=' not in line:
            raise ValueError(
                f'Строка {line_number}: ожидается запись КЛЮЧ=значение'
            )

        key, value = line.split('=', 1)
        credentials[key.strip()] = value.strip()

    return credentials

credentials = read_credentials(CREDENTIALS_PATH)

sphere_required = [
    'SPHERE_HOST',
    'SPHERE_DATABASE',
    'SPHERE_USER',
    'SPHERE_PASSWORD',
]
sphere_missing = [key for key in sphere_required if not credentials.get(key)]
if sphere_missing:
    raise ValueError(
        'Заполни в уч данные.txt: ' + ', '.join(sphere_missing)
    )

SPHERE_HOST = credentials['SPHERE_HOST']
SPHERE_PORT = int(credentials.get('SPHERE_PORT', '5432'))
SPHERE_DATABASE = credentials['SPHERE_DATABASE']
SPHERE_USER = credentials['SPHERE_USER']
SPHERE_PASSWORD = credentials['SPHERE_PASSWORD']

connection_url = URL.create(
    drivername='postgresql+psycopg',
    username=SPHERE_USER,
    password=SPHERE_PASSWORD,
    host=SPHERE_HOST,
    port=SPHERE_PORT,
    database=SPHERE_DATABASE,
)
engine = create_engine(connection_url, pool_pre_ping=True)

print('Учётные данные прочитаны, подключение к Сфере создано')

Учётные данные прочитаны, подключение к Сфере создано


In [5]:
with engine.connect() as connection:
    connection_check = pd.read_sql_query(
        text('select current_database() as database_name, current_user as user_name'),
        connection,
    )

display(connection_check)

,database_name,user_name
0,postgres,svetovavs


# 2. Подключение к Oracle КХД



In [6]:
khd_required = [
    'KHD_HOST',
    'KHD_SERVICE_NAME',
    'KHD_USER',
    'KHD_PASSWORD',
]
khd_missing = [key for key in khd_required if not credentials.get(key)]
if khd_missing:
    raise ValueError(
        'Заполни в уч данные.txt: ' + ', '.join(khd_missing)
    )

KHD_HOST = credentials['KHD_HOST']
KHD_PORT = int(credentials.get('KHD_PORT', '1521'))
KHD_SERVICE_NAME = credentials['KHD_SERVICE_NAME']
KHD_USER = credentials['KHD_USER']
KHD_PASSWORD = credentials['KHD_PASSWORD']
KHD_DATA_SCHEMA = credentials.get('KHD_DATA_SCHEMA', 'DM_RISK_AVATAR')

khd_dsn = oracledb.makedsn(
    KHD_HOST,
    KHD_PORT,
    service_name=KHD_SERVICE_NAME,
)
khd_connection = oracledb.connect(
    user=KHD_USER,
    password=KHD_PASSWORD,
    dsn=khd_dsn,
)

print('Подключение к КХД создано')


Подключение к КХД создано


In [7]:
with khd_connection.cursor() as cursor:
    cursor.execute(
        "select user, sys_context('USERENV', 'DB_NAME') from dual"
    )
    khd_user_name, khd_database_name = cursor.fetchone()

    cursor.execute(
        """
        select table_name
        from all_tables
        where owner = :owner
          and table_name = 'EGRN_DATA'
        order by table_name
        """,
        owner=KHD_DATA_SCHEMA.upper(),
    )
    available_khd_tables = [row[0] for row in cursor.fetchall()]

print('Пользователь КХД:', khd_user_name)
print('База КХД:', khd_database_name)
print('Доступные таблицы:', available_khd_tables)

expected_khd_tables = {'EGRN_DATA'}
missing_khd_tables = sorted(expected_khd_tables - set(available_khd_tables))
if missing_khd_tables:
    raise PermissionError(
        'Не видны таблицы КХД: ' + ', '.join(missing_khd_tables)
    )

with khd_connection.cursor() as cursor:
    cursor.execute(
        """
        select column_name
        from all_tab_columns
        where owner = :owner
          and table_name = 'EGRN_DATA'
        order by column_id
        """,
        owner=KHD_DATA_SCHEMA.upper(),
    )
    egrn_source_columns = [row[0] for row in cursor.fetchall()]

if not egrn_source_columns:
    raise PermissionError('Не удалось получить список колонок EGRN_DATA')

print('Колонок в EGRN_DATA:', len(egrn_source_columns))


Пользователь КХД: SVETOVAVS
База КХД: ODSPROD
Доступные таблицы: ['EGRN_DATA']
Колонок в EGRN_DATA: 142


# 3. SQL Сфера, расширенный подход


In [8]:
expanded_sql = r"""
/*
запускать в сфере

запрос собирает все неудаленные объекты недвижимости
если объект связан с подходящим договором данные договора заполняются
если связь не найдена объект остается в результате с пустыми полями договора

одна строка для связанного объекта означает объект в одном договоре
одна строка для несвязанного объекта означает его последнюю версию характеристик
*/

with task_candidates as (
    /* отбираем подходящие задачи оформления */
    select
        t.id as task_id,
        r.id as request_id,
        c.id as contract_id,
        row_number() over (
            partition by c.id
            order by
                coalesce(
                    t.d_conclusion_ins_contract::timestamp with time zone,
                    t.d_create,
                    t.d_change
                ) desc nulls last,
                t.d_create desc nulls last,
                t.d_change desc nulls last,
                t.id desc
        ) as task_number
    from bps_request_ins_task t
    join bps_request_ins r
        on r.id = t.request_ins_id
    join bps_contract c
        on c.id = r.contract_id
    where t.task_type = 'draft_contract'
      and t.status = 'operational_archive'
      and (
          t.ins_document_type = 'new_ins_contract'
          or t.ins_document_type = 'ins_contract_prolong'
          or t.ins_document_type is null
      )
      and t.ins_refuse is not true
      and t.d_delete is null
      and r.d_delete is null
      and c.d_delete is null
),

selected_tasks as (
    /* оставляем последнюю подходящую задачу каждого договора */
    select
        task_id,
        request_id,
        contract_id
    from task_candidates
    where task_number = 1
),

linked_object_candidates as (
    /* находим недвижимость в выбранных задачах */
    select
        ch.insurance_object_id as object_id,
        ch.id as characteristics_id,
        link.id as task_object_link_id,
        selected.task_id,
        selected.request_id,
        selected.contract_id,
        row_number() over (
            partition by selected.task_id, ch.insurance_object_id
            order by
                link.d_change desc nulls last,
                link.d_create desc nulls last,
                ch.version_start_date desc nulls last,
                ch.version_number desc nulls last,
                link.id desc
        ) as link_number
    from selected_tasks selected
    join bps_request_ins_task_insurance_object link
        on link.parent_id = selected.task_id
    join base_insurance_object_characteristics ch
        on ch.id = link.characteristics_id
    join base_insurance_object obj
        on obj.id = ch.insurance_object_id
    where obj.elementary_obj_type = 'nedv_ul_and_ip'
      and obj.d_delete is null
),

selected_links as (
    /* убираем повторные связи одного объекта с одной задачей */
    select
        object_id,
        characteristics_id,
        task_object_link_id,
        task_id,
        request_id,
        contract_id
    from linked_object_candidates
    where link_number = 1
),

object_versions as (
    /* нумеруем версии характеристик каждого объекта */
    select
        obj.id as object_id,
        ch.id as characteristics_id,
        row_number() over (
            partition by obj.id
            order by
                ch.version_is_active desc nulls last,
                ch.version_number desc nulls last,
                ch.version_start_date desc nulls last,
                ch.id desc nulls last
        ) as version_number
    from base_insurance_object obj
    left join base_insurance_object_characteristics ch
        on ch.insurance_object_id = obj.id
    where obj.elementary_obj_type = 'nedv_ul_and_ip'
      and obj.d_delete is null
),

dataset_keys as (
    /* сохраняем все найденные связи с договорами */
    select
        linked.object_id,
        linked.characteristics_id,
        linked.task_object_link_id,
        linked.task_id,
        linked.request_id,
        linked.contract_id,
        'linked'::text as row_source
    from selected_links linked

    union all

    /* добавляем объекты для которых подходящий договор не найден */
    select
        version.object_id,
        version.characteristics_id,
        null::integer as task_object_link_id,
        null::integer as task_id,
        null::integer as request_id,
        null::integer as contract_id,
        'not_linked'::text as row_source
    from object_versions version
    where version.version_number = 1
      and not exists (
          select 1
          from selected_links linked
          where linked.object_id = version.object_id
      )
),

object_link_profile as (
    /* считаем со сколькими договорами связан объект */
    select
        object_id,
        count(distinct contract_id) as contract_count
    from selected_links
    group by object_id
),

selected_characteristics as (
    /* ограничиваем расчет условий версиями из итоговой выборки */
    select distinct characteristics_id
    from dataset_keys
    where characteristics_id is not null
),

condition_summary as (
    /* сворачиваем варианты условий в одну строку */
    select
        cond.characteristics_id,
        count(*) as condition_count,
        count(cond.insured_sum) as filled_insured_sum_count,
        count(distinct cond.insured_sum) filter (
            where cond.insured_sum is not null
        ) as distinct_insured_sum_count,
        min(cond.insured_sum) as minimum_insured_sum,
        max(cond.insured_sum) as maximum_insured_sum,
        count(distinct cond.insured_sum_currency) filter (
            where cond.insured_sum_currency is not null
        ) as currency_count,
        string_agg(
            distinct cond.insured_sum_currency,
            ', '
            order by cond.insured_sum_currency
        ) filter (
            where cond.insured_sum_currency is not null
        ) as insured_sum_currency,
        array_agg(
            distinct cond.terms_option_number
            order by cond.terms_option_number
        ) filter (
            where cond.terms_option_number is not null
        ) as terms_option_numbers,
        min(cond.per_occurance_limit) as minimum_per_occurrence_limit,
        max(cond.per_occurance_limit) as maximum_per_occurrence_limit,
        jsonb_agg(
            to_jsonb(cond)
            order by cond.id
        ) as source_json__base_insurance_object_conditions,
        case
            when count(distinct cond.insured_sum) filter (
                where cond.insured_sum is not null
            ) = 1
             and count(distinct cond.insured_sum_currency) filter (
                where cond.insured_sum_currency is not null
            ) <= 1
            then max(cond.insured_sum)
        end as insured_sum
    from base_insurance_object_conditions cond
    join selected_characteristics selected
        on selected.characteristics_id = cond.characteristics_id
    group by cond.characteristics_id
),

raw_result as (
/* собираем исходные поля расширенного датасета */
select
    /* качество строки */
    keys.row_source,
    case
        when coalesce(profile.contract_count, 0) = 0 then 'not_linked'
        when profile.contract_count = 1 then 'linked'
        else 'multiple_contracts'
    end as contract_link_status,
    coalesce(profile.contract_count, 0) as contract_count,
    (keys.contract_id is not null) as has_contract,
    (address.id is not null) as has_address,
    (conditions.insured_sum is not null) as has_target,
    case
        when conditions.condition_count is null then 'no_conditions'
        when conditions.filled_insured_sum_count = 0 then 'target_is_empty'
        when conditions.distinct_insured_sum_count > 1 then 'several_target_values'
        when conditions.currency_count > 1 then 'several_currencies'
        when conditions.insured_sum <= 0 then 'target_is_not_positive'
        else 'target_is_usable'
    end as target_status,

    /* идентификаторы */
    keys.object_id,
    keys.characteristics_id,
    keys.task_object_link_id,
    keys.task_id,
    keys.request_id,
    keys.contract_id,
    obj.geo_address_id,
    contract.contractor_id as policyholder_id,
    request.corporate_crm_id,

    /* целевая страховая сумма */
    conditions.insured_sum,
    conditions.insured_sum_currency,
    conditions.condition_count,
    conditions.filled_insured_sum_count,
    conditions.distinct_insured_sum_count,
    conditions.minimum_insured_sum as condition_min_insured_sum,
    conditions.maximum_insured_sum as condition_max_insured_sum,
    conditions.currency_count as condition_currency_count,
    conditions.terms_option_numbers,

    /* контрольные суммы */
    task_link.insured_sum as task_object_insured_sum,
    task_link.insured_sum_currency as task_object_insured_sum_currency,
    task.total_ins_contract_amount as contract_insured_sum,
    task.curr_ins_contract_amount as contract_amount_currency,
    task.total_ins_contract_premium as contract_premium,
    ch.insurance_value,
    ch.insurance_value_currency,
    ch.insurance_value_basis,
    ch.pledged_value,
    conditions.minimum_per_occurrence_limit,
    conditions.maximum_per_occurrence_limit,

    /* объект */
    obj.obj_type as object_type,
    obj.elementary_obj_type,
    obj.obj_name as object_name,
    obj.description as object_description,
    obj.original_address,
    obj.active as object_is_active,
    obj.d_create as object_create_date,
    obj.d_change as object_change_date,

    /* характеристики объекта */
    ch.version_number as characteristics_version_number,
    ch.version_start_date as characteristics_version_start_date,
    ch.version_end_date as characteristics_version_end_date,
    ch.version_is_active as characteristics_version_is_active,
    ch.ownership_type,
    ch.is_pledged,
    ch.is_leased,
    ch.insured_components,
    ch.activity_types,
    ch.risk_natures,
    ch.insurance_territory,
    ch.has_losses,
    ch.insurance_object_loss_history,
    ch.characteristics ->> 'total_area_sq_m' as total_area,
    ch.characteristics ->> 'occupied_area_sq_m' as occupied_area,
    ch.characteristics ->> 'construction_year' as construction_year,
    ch.characteristics ->> 'last_capital_repair_year' as capital_repair_year,
    ch.characteristics ->> 'total_floors_count' as floors_count,
    ch.characteristics ->> 'occupied_floor' as occupied_floor,
    ch.characteristics ->> 'load_bearing_walls_material' as walls_material,
    ch.characteristics ->> 'interfloor_overlap_material' as overlap_material,
    ch.characteristics ->> 'roofing_material' as roofing_material,
    ch.characteristics ->> 'fire_alarm_system_availability'
        as fire_alarm_system_availability,
    ch.characteristics ->> 'fire_suppression_system_availability'
        as fire_suppression_system_availability,
    ch.characteristics ->> 'nearest_fire_station_distance_km'
        as nearest_fire_station_distance_km,
    ch.characteristics as object_characteristics_json,

    /* адрес */
    address.full_address,
    address.postal_code,
    address.region_id as address_region_id,
    address.area as district,
    address.settlement_type,
    address.settlement,
    address.street_type,
    address.street,
    address.house,
    address.building,
    address.block,
    address.flat,
    address.office,
    address.fias_code,
    address.longitude,
    address.latitude,
    address.address_dgis_id,

    /* договор */
    contract.n_contract as contract_number,
    contract.document_status as contract_status,
    contract.system_type as contract_source_system,
    contract.ins_product_sbs as insurance_product,
    contract.d_sign_contract as contract_sign_date,
    contract.d_start_contract as contract_start_date,
    contract.d_end_contract as contract_end_date,
    contract.prevcontract_id as previous_contract_id,
    contract.rootcontract_id as root_contract_id,
    previous_contract.n_contract as previous_contract_number,
    previous_contract.d_start_contract as previous_contract_start_date,
    previous_contract.d_end_contract as previous_contract_end_date,

    /* задача и заявка */
    task.task_type,
    task.status as task_status,
    task.ins_document_type,
    task.ins_refuse,
    task.d_create as task_create_date,
    task.d_conclusion_ins_contract as contract_conclusion_date,
    task.ins_product as task_product,
    task.industry as task_industry,
    task.subindustry as task_subindustry,
    task.locations_count,
    task.multi_location,
    task.object_description as task_object_description,
    request.business_segment,
    request.sale_channel,
    request.ins_product as request_product,

    /* страхователь и crm */
    policyholder.inn as policyholder_inn,
    policyholder.company_name_short as policyholder_name,
    policyholder.cdi_id as policyholder_cdi_id,
    crm.segment as crm_segment,
    crm.macroindustry as crm_macroindustry,
    crm.industry as crm_industry,
    crm.okved as crm_okved,

    /* полные строки таблиц для статистического исследования */
    to_jsonb(obj) as source_json__base_insurance_object,
    to_jsonb(ch) as source_json__base_insurance_object_characteristics,
    conditions.source_json__base_insurance_object_conditions,
    to_jsonb(address) as source_json__base_geo_address,
    to_jsonb(task_link) as source_json__bps_request_ins_task_insurance_object,
    to_jsonb(task) as source_json__bps_request_ins_task,
    to_jsonb(request) as source_json__bps_request_ins,
    to_jsonb(contract) as source_json__bps_contract,
    to_jsonb(previous_contract) as source_json__bps_contract_previous,
    to_jsonb(policyholder) as source_json__bps_contractor,
    to_jsonb(crm) as source_json__bps_corporate_crm,

    /* дата состояния строки */
    coalesce(
        task.d_conclusion_ins_contract::timestamp with time zone,
        contract.d_sign_contract,
        ch.version_start_date,
        obj.d_create
    ) as as_of_date
from dataset_keys keys
join base_insurance_object obj
    on obj.id = keys.object_id
left join base_insurance_object_characteristics ch
    on ch.id = keys.characteristics_id
left join condition_summary conditions
    on conditions.characteristics_id = keys.characteristics_id
left join bps_request_ins_task_insurance_object task_link
    on task_link.id = keys.task_object_link_id
left join bps_request_ins_task task
    on task.id = keys.task_id
left join bps_request_ins request
    on request.id = keys.request_id
left join bps_contract contract
    on contract.id = keys.contract_id
left join bps_contract previous_contract
    on previous_contract.id = contract.prevcontract_id
left join bps_contractor policyholder
    on policyholder.id = contract.contractor_id
left join bps_corporate_crm crm
    on crm.id = request.corporate_crm_id
left join base_geo_address address
    on address.id = obj.geo_address_id
left join object_link_profile profile
    on profile.object_id = keys.object_id
),

standardized_result as (
    /* приводим результат к общей структуре двух датасетов */
    select
        case
            when raw.contract_id is null then 'not_linked'
            else 'linked'
        end as row_source,
        case
            when count(raw.contract_id) over (
                partition by raw.object_id
            ) = 0 then 'not_linked'
            when count(raw.contract_id) over (
                partition by raw.object_id
            ) = 1 then 'linked'
            else 'multiple_contracts'
        end as contract_link_status,
        count(raw.contract_id) over (
            partition by raw.object_id
        ) as contract_count,
        (raw.contract_id is not null) as has_contract,
        (
            raw.geo_address_id is not null
            or nullif(btrim(raw.full_address), '') is not null
            or nullif(btrim(raw.original_address), '') is not null
        ) as has_address,
        (raw.insured_sum is not null) as has_target,
        case
            when raw.condition_count is null
              or raw.condition_count = 0
                then 'no_conditions'
            when raw.condition_min_insured_sum is distinct from
                 raw.condition_max_insured_sum
                then 'several_target_values'
            when coalesce(raw.condition_currency_count, 0) > 1
                then 'several_currencies'
            when raw.insured_sum <= 0
                then 'target_is_not_positive'
            when raw.insured_sum is null
                then 'target_is_empty'
            else 'target_is_usable'
        end as target_status,

        raw.contract_id,
        raw.contract_number,
        raw.previous_contract_id,
        raw.root_contract_id,
        raw.request_id,
        raw.task_id,
        raw.task_object_link_id,
        raw.characteristics_id,
        raw.object_id,
        raw.geo_address_id,
        raw.policyholder_id,
        raw.corporate_crm_id,

        raw.as_of_date,
        raw.contract_conclusion_date,
        raw.contract_sign_date,
        raw.contract_start_date,
        raw.contract_end_date,
        raw.contract_status,
        raw.ins_document_type,
        raw.insurance_product,

        case
            when raw.contract_id is not null then
                count(raw.object_id) over (
                    partition by raw.contract_id
                )
        end as real_estate_objects_in_contract,
        raw.object_name,
        raw.object_description,
        raw.object_type,
        raw.elementary_obj_type,
        raw.total_area,
        raw.occupied_area,
        raw.construction_year,
        raw.capital_repair_year,
        raw.floors_count,
        raw.occupied_floor,
        raw.walls_material,
        raw.overlap_material,
        raw.roofing_material,
        raw.ownership_type,
        raw.is_leased,
        raw.insured_components,
        raw.activity_types,
        raw.risk_natures,
        raw.insurance_territory,

        raw.full_address,
        raw.original_address,
        raw.postal_code,
        raw.address_region_id,
        raw.district,
        raw.settlement,
        raw.street,
        raw.house,
        raw.building,
        raw.block,
        raw.flat,
        raw.office,
        raw.fias_code,
        raw.longitude,
        raw.latitude,
        raw.address_dgis_id,

        raw.policyholder_inn,
        raw.policyholder_name,
        raw.policyholder_cdi_id,
        raw.crm_segment,
        raw.crm_macroindustry,
        raw.crm_industry,
        raw.crm_okved,
        raw.business_segment,
        raw.task_industry,
        raw.task_subindustry,

        raw.contract_insured_sum,
        raw.contract_amount_currency,
        raw.task_object_insured_sum,
        raw.task_object_insured_sum_currency,
        raw.condition_min_insured_sum,
        raw.condition_max_insured_sum,
        raw.insured_sum,
        raw.insured_sum_currency,
        raw.condition_currency_count,
        raw.contract_premium,
        raw.insurance_value,
        raw.insurance_value_currency,
        raw.insurance_value_basis,
        raw.is_pledged,
        raw.pledged_value,
        raw.minimum_per_occurrence_limit,
        raw.maximum_per_occurrence_limit,

        raw.previous_contract_number,
        raw.previous_contract_start_date,
        raw.previous_contract_end_date,

        raw.condition_count,
        raw.characteristics_version_number,
        raw.characteristics_version_start_date,
        raw.characteristics_version_end_date,
        raw.characteristics_version_is_active,
        raw.object_characteristics_json,
        raw.task_type,
        raw.task_status,
        raw.ins_refuse,
        raw.source_json__base_insurance_object,
        raw.source_json__base_insurance_object_characteristics,
        raw.source_json__base_insurance_object_conditions,
        raw.source_json__base_geo_address,
        raw.source_json__bps_request_ins_task_insurance_object,
        raw.source_json__bps_request_ins_task,
        raw.source_json__bps_request_ins,
        raw.source_json__bps_contract,
        raw.source_json__bps_contract_previous,
        raw.source_json__bps_contractor,
        raw.source_json__bps_corporate_crm
    from raw_result raw
)

select *
from standardized_result
order by
    has_contract desc,
    as_of_date desc nulls last,
    object_id;

"""


In [9]:
with engine.connect() as connection:
    expanded_df = pd.read_sql_query(text(expanded_sql), connection)

print('Строк:', len(expanded_df))
print('Колонок:', len(expanded_df.columns))
print('Данные Сферы загружены')

# разворачиваем все поля исходных таблиц в отдельные колонки
def parse_json_container(value):
    if value is None or (not isinstance(value, (dict, list)) and pd.isna(value)):
        return None
    if isinstance(value, (dict, list)):
        return value
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return value
    return value


def one_value_or_json(values):
    clean_values = []
    seen = set()
    for value in values:
        if value is None or (not isinstance(value, (dict, list)) and pd.isna(value)):
            continue
        marker = json.dumps(value, ensure_ascii=False, sort_keys=True, default=str)
        if marker not in seen:
            seen.add(marker)
            clean_values.append(value)
    if not clean_values:
        return None
    if len(clean_values) == 1:
        value = clean_values[0]
        if isinstance(value, (dict, list)):
            return json.dumps(value, ensure_ascii=False, sort_keys=True, default=str)
        return value
    return json.dumps(clean_values, ensure_ascii=False, sort_keys=True, default=str)


json_columns = [
    column for column in expanded_df.columns
    if column.startswith('source_json__')
]

for json_column in json_columns:
    table_name = json_column.removeprefix('source_json__')
    containers = expanded_df[json_column].map(parse_json_container)

    if table_name == 'base_insurance_object_conditions':
        condition_keys = sorted({
            key
            for rows in containers.dropna()
            if isinstance(rows, list)
            for row in rows
            if isinstance(row, dict)
            for key in row
        })
        normalized = pd.DataFrame(index=expanded_df.index)
        for key in condition_keys:
            normalized[f'sphere__{table_name}__{key}'] = containers.map(
                lambda rows: one_value_or_json([
                    row.get(key)
                    for row in rows
                    if isinstance(row, dict)
                ]) if isinstance(rows, list) else None
            )
    else:
        records = containers.map(
            lambda value: value if isinstance(value, dict) else {}
        ).tolist()
        normalized = pd.json_normalize(records, sep='__')
        normalized.index = expanded_df.index
        normalized.columns = [
            f'sphere__{table_name}__{column}'
            for column in normalized.columns
        ]

    expanded_df = pd.concat(
        [expanded_df.drop(columns=[json_column]), normalized],
        axis=1,
    )

if expanded_df.columns.duplicated().any():
    duplicates = expanded_df.columns[expanded_df.columns.duplicated()].tolist()
    raise ValueError('После раскрытия таблиц появились повторные колонки: ' + ', '.join(duplicates))

print('Колонок после раскрытия всех полей Сферы:', len(expanded_df.columns))


Строк: 14652
Колонок: 113
Данные Сферы загружены
Колонок после раскрытия всех полей Сферы: 861


# 4. Проверка заполненности


In [10]:
required_columns = {
    'row_source', 'object_id', 'characteristics_id', 'contract_id',
    'elementary_obj_type', 'has_contract', 'has_address', 'has_target',
    'target_status'
}
missing_columns = sorted(required_columns - set(expanded_df.columns))
if missing_columns:
    raise ValueError('Не найдены ожидаемые колонки: ' + ', '.join(missing_columns))

profile = pd.DataFrame({
    'Показатель': [
        'Строк',
        'Уникальных объектов',
        'Уникальных характеристик',
        'Уникальных договоров',
        'Строк с договором',
        'Строк с адресом',
        'Строк с target',
        'Строк с пустым типом объекта',
    ],
    'Значение': [
        len(expanded_df),
        expanded_df['object_id'].nunique(dropna=True),
        expanded_df['characteristics_id'].nunique(dropna=True),
        expanded_df['contract_id'].nunique(dropna=True),
        expanded_df['has_contract'].fillna(False).astype(bool).sum(),
        expanded_df['has_address'].fillna(False).astype(bool).sum(),
        expanded_df['has_target'].fillna(False).astype(bool).sum(),
        expanded_df['elementary_obj_type'].fillna('').str.strip().eq('').sum(),
    ],
})

display(profile)

,Показатель,Значение
0,Строк,14652
1,Уникальных объектов,14642
2,Уникальных характеристик,14642
3,Уникальных договоров,569
4,Строк с договором,1206
5,Строк с адресом,12974
6,Строк с target,13469
7,Строк с пустым типом объекта,0


In [11]:
display(expanded_df['row_source'].fillna('empty').value_counts(dropna=False))
display(expanded_df['elementary_obj_type'].fillna('empty').value_counts(dropna=False))
display(expanded_df['target_status'].fillna('empty').value_counts(dropna=False))

row_source
not_linked    13446
linked         1206
Name: count, dtype: int64

elementary_obj_type
nedv_ul_and_ip    14652
Name: count, dtype: int64

target_status
target_is_usable          13427
target_is_empty             688
no_conditions               451
several_target_values        44
target_is_not_positive       42
Name: count, dtype: int64

# 5. Соединение с ЕГРН

Объекты передаются в один Oracle `SELECT` как JSON-параметр. В КХД ничего не создаётся и не записывается. Поиск выполняется на уровне здания. Данные ЕГРН присоединяются только при одном кандидате; неоднозначные случаи остаются пустыми.


In [12]:
# готовим только те поля, которые нужны Oracle для поиска ЕГРН
sphere_with_row_id = expanded_df.copy()
sphere_with_row_id.insert(0, 'sphere_row_id', range(1, len(sphere_with_row_id) + 1))

stage_columns = [
    'sphere_row_id',
    'contract_id',
    'contract_number',
    'task_id',
    'task_object_link_id',
    'characteristics_id',
    'object_id',
    'geo_address_id',
    'real_estate_objects_in_contract',
    'object_description',
    'total_area',
    'full_address',
    'original_address',
    'postal_code',
    'settlement',
    'street',
    'house',
    'building',
    'block',
    'flat',
    'office',
]

missing_stage_columns = [
    column for column in stage_columns
    if column not in sphere_with_row_id.columns
]
if missing_stage_columns:
    raise ValueError(
        'Для поиска ЕГРН не хватает колонок: '
        + ', '.join(missing_stage_columns)
    )

def json_value(column, value):
    if value is None or pd.isna(value):
        return None
    if column == 'sphere_row_id':
        return int(value)
    return str(value)

sphere_records = []
for row in sphere_with_row_id[stage_columns].itertuples(index=False, name=None):
    sphere_records.append({
        column: json_value(column, value)
        for column, value in zip(stage_columns, row)
    })

sphere_json = json.dumps(sphere_records, ensure_ascii=False)
print('Строк передано в Oracle SELECT:', len(sphere_records))
print('Размер JSON, МБ:', round(len(sphere_json.encode('utf-8')) / 1024**2, 2))


Строк передано в Oracle SELECT: 14652
Размер JSON, МБ: 8.63


In [13]:
egrn_sql = r"""
/*
Соединение временной выборки объектов Сферы с ЕГРН по адресу.

Запускать в Oracle.

Результат содержит одну строку на объект Сферы. Данные ЕГРН заполняются
только тогда, когда по адресу найдена одна запись уровня здания.

Для поиска используются населённый пункт, улица и дом. Корпус и строение
учитываются, если они указаны.
Индекс и регион сравниваются, если они заполнены с обеих сторон.
Если по адресу найдено несколько зданий, площадь используется как
дополнительная проверка. Если площади нет или она не помогла, кандидаты
по адресу не отсекаются.

В поиск попадают только типы ЕГРН «здание», «сооружение» и «строение»
с уровнем адреса FIAS_HOUSE. Квартиры, офисы, комнаты и помещения исключены.
Если по одному адресу Сферы записано несколько страховых объектов,
одно найденное здание ЕГРН присоединяется к каждому из них.

Источник передаётся из notebook одним JSON-параметром `sphere_json`.
Запрос только читает EGRN_DATA и ничего не создаёт в Oracle.
*/

with sphere_source as (
    /* Берём только поля, которые нужны для проверки соединения. */
    select /*+ materialize */
        s.sphere_row_id,
        s.contract_id,
        s.contract_number,
        s.task_id,
        s.task_object_link_id,
        s.characteristics_id,
        s.object_id,
        s.geo_address_id,
        s.real_estate_objects_in_contract,
        cast(s.object_description as varchar2(4000))
            as object_description,
        s.total_area,
        cast(s.full_address as varchar2(4000)) as full_address,
        cast(s.original_address as varchar2(4000)) as original_address,
        s.postal_code,
        s.settlement,
        s.street,
        s.house,
        s.building,
        s.block,
        s.flat,
        s.office
    from json_table(
        :sphere_json,
        '$[*]'
        columns (
            sphere_row_id number path '$.sphere_row_id',
            contract_id varchar2(200) path '$.contract_id',
            contract_number varchar2(500) path '$.contract_number',
            task_id varchar2(200) path '$.task_id',
            task_object_link_id varchar2(200)
                path '$.task_object_link_id',
            characteristics_id varchar2(200)
                path '$.characteristics_id',
            object_id varchar2(200) path '$.object_id',
            geo_address_id varchar2(200) path '$.geo_address_id',
            real_estate_objects_in_contract varchar2(200)
                path '$.real_estate_objects_in_contract',
            object_description varchar2(4000)
                path '$.object_description',
            total_area varchar2(200) path '$.total_area',
            full_address varchar2(4000) path '$.full_address',
            original_address varchar2(4000) path '$.original_address',
            postal_code varchar2(100) path '$.postal_code',
            settlement varchar2(1000) path '$.settlement',
            street varchar2(1000) path '$.street',
            house varchar2(500) path '$.house',
            building varchar2(500) path '$.building',
            block varchar2(500) path '$.block',
            flat varchar2(500) path '$.flat',
            office varchar2(500) path '$.office'
        )
    ) s
),

sphere_text as (
    /* Если нормализованного адреса нет, используем адрес, введённый вручную. */
    select
        s.*,
        replace(
            regexp_replace(trim(s.total_area), '[[:space:]]+', ''),
            ',',
            '.'
        ) as sphere_area_text,
        coalesce(
            nullif(trim(s.full_address), ''),
            nullif(trim(s.original_address), '')
        ) as source_address,
        regexp_replace(
            regexp_replace(
                replace(
                    lower(
                        replace(
                            coalesce(
                                nullif(trim(s.full_address), ''),
                                nullif(trim(s.original_address), '')
                            ),
                            chr(160),
                            ' '
                        )
                    ),
                    'ё',
                    'е'
                ),
                '[;|]+',
                ','
            ),
            '[[:space:]]*,[[:space:]]*',
            ', '
        ) as address_text
    from sphere_source s
),

sphere_parts_raw as (
    /* Берём готовые части адреса, а при их отсутствии разбираем полный адрес. */
    select
        s.*,
        coalesce(
            nullif(trim(s.postal_code), ''),
            regexp_substr(
                s.address_text,
                '(^|,)[[:space:]]*([0-9]{6})([[:space:]]*,|$)',
                1, 1, 'i', 2
            )
        ) as postal_code_raw,
        regexp_substr(
            s.address_text,
            '(^|,)[[:space:]]*([^,]*(область|обл[.]?|край|республика|респ[.]?)[^,]*)',
            1, 1, 'i', 2
        ) as region_raw,
        coalesce(
            nullif(trim(s.settlement), ''),
            regexp_substr(
                s.address_text,
                '^[[:space:]]*([^,(]+)[[:space:]]*[(]',
                1, 1, 'i', 1
            ),
            regexp_substr(
                s.address_text,
                '(^|,)[[:space:]]*(пгт|поселок городского типа|рабочий поселок|р[.]?п|поселок|пос|село|с|деревня|д|хутор|х)[.]?[[:space:]]+([^,]+)',
                1, 1, 'i', 3
            ),
            regexp_substr(
                s.address_text,
                '(^|,)[[:space:]]*(город|г)[.]?[[:space:]]+([^,]+)',
                1, 1, 'i', 3
            ),
            regexp_substr(
                s.address_text,
                '^[[:space:]]*([^,(]+)[[:space:]]*,[[:space:]]*(аэропорт|аэродром)([[:space:]]+|,|$)',
                1, 1, 'i', 1
            ),
            regexp_substr(
                s.address_text,
                '^[[:space:]]*([^,(]+)[[:space:]]*,[[:space:]]*(улица|ул[.]?|проспект|пр-кт|переулок|пер[.]?|шоссе|ш[.]?|набережная|наб[.]?|бульвар|б-р|проезд|площадь|пл[.]?|тракт|аллея|микрорайон|мкр[.]?)[[:space:]]+',
                1, 1, 'i', 1
            ),
            regexp_substr(
                s.address_text,
                '(^|,)[[:space:]]*([^,]+)[[:space:]]*,[[:space:]]*([^,]*(улица|ул[.]?|проспект|пр-кт|переулок|пер[.]?|шоссе|ш[.]?|набережная|наб[.]?|бульвар|б-р|проезд|площадь|пл[.]?|тракт|аллея|микрорайон|мкр[.]?))([[:space:]]*,|$)',
                1, 1, 'i', 2
            )
        ) as locality_raw,
        coalesce(
            nullif(trim(s.street), ''),
            regexp_substr(
                s.address_text,
                '(^|,)[[:space:]]*(улица|ул[.]?|проспект|пр-кт|переулок|пер[.]?|шоссе|ш[.]?|набережная|наб[.]?|бульвар|б-р|проезд|площадь|пл[.]?|тракт|аллея|микрорайон|мкр[.]?)[[:space:]]+([^,]+)',
                1, 1, 'i', 3
            ),
            regexp_substr(
                s.address_text,
                '(^|,)[[:space:]]*([^,]+)[[:space:]]+(улица|ул[.]?|проспект|пр-кт|переулок|пер[.]?|шоссе|ш[.]?|набережная|наб[.]?|бульвар|б-р|проезд|площадь|пл[.]?|тракт|аллея|микрорайон|мкр[.]?)([[:space:]]*,|$)',
                1, 1, 'i', 2
            )
        ) as street_raw,
        coalesce(
            nullif(trim(s.house), ''),
            regexp_substr(
                s.address_text,
                '(^|,)[[:space:]]*(дом|д)[.]?[[:space:]]*([0-9]+[а-яa-z]?([/-][0-9а-яa-z]+)?)',
                1, 1, 'i', 3
            ),
            regexp_substr(
                s.address_text,
                '(^|,)[[:space:]]*([0-9]{1,5}[а-яa-z]?([/-][0-9а-яa-z]+)?)[[:space:]]+(корпус|корп|к|строение|стр|ст)[.]?[[:space:]]*[0-9а-яa-z/-]+([[:space:]]*,|$)',
                1, 1, 'i', 2
            ),
            regexp_substr(
                s.address_text,
                '(^|,)[[:space:]]*([0-9]{1,5}[а-яa-z]?([/-][0-9а-яa-z]+)?)([[:space:]]*,|$)',
                1, 1, 'i', 2
            )
        ) as house_raw,
        coalesce(
            nullif(trim(s.block), ''),
            regexp_substr(
                s.address_text,
                '(^|,)[[:space:]]*(дом|д)[.]?[[:space:]]*[0-9а-яa-z/-]+[[:space:]]*(корпус|корп|к)[.]?[[:space:]]*([0-9а-яa-z/-]+)',
                1, 1, 'i', 4
            ),
            regexp_substr(
                s.address_text,
                '(^|,|[[:space:]])(корпус|корп|к)[.]?[[:space:]]*([0-9а-яa-z/-]+)',
                1, 1, 'i', 3
            )
        ) as korpus_raw,
        coalesce(
            nullif(trim(s.building), ''),
            regexp_substr(
                s.address_text,
                '(^|,|[[:space:]])(строение|стр|ст)[.]?[[:space:]]*([0-9а-яa-z/-]+)',
                1, 1, 'i', 3
            )
        ) as stroenie_raw,
        regexp_substr(
            s.address_text,
            '(^|,)[[:space:]]*((аэропорт|аэродром)[^,]*)',
            1, 1, 'i', 2
        ) as landmark_raw
    from sphere_text s
),

sphere_prepared as (
    /* Приводим части адреса к одному виду для сравнения. */
    select /*+ materialize */
        s.*,
        case
            when regexp_like(
                s.sphere_area_text,
                '^[0-9]+([.][0-9]+)?$'
            )
            then to_number(
                s.sphere_area_text,
                '999999999999999999999999D9999999999',
                'NLS_NUMERIC_CHARACTERS=''.,'''
            )
        end as sphere_area,
        regexp_replace(s.postal_code_raw, '[^0-9]+', '')
            as sphere_postal_code,
        regexp_replace(
            regexp_replace(
                replace(lower(trim(s.region_raw)), 'ё', 'е'),
                '(^|[[:space:]])(область|обл|край|республика|респ)([.]|[[:space:]]|$)',
                ' '
            ),
            '[^[:alnum:]]+',
            ''
        ) as sphere_region,
        regexp_replace(
            regexp_replace(
                replace(lower(trim(s.locality_raw)), 'ё', 'е'),
                '(^|[[:space:]])(город|г|пгт|поселок городского типа|рабочий поселок|р[.]?п|поселок|пос|село|с|деревня|д|хутор|х)([.]|[[:space:]]|$)',
                ' '
            ),
            '[^[:alnum:]]+',
            ''
        ) as sphere_locality,
        regexp_replace(
            regexp_replace(
                replace(lower(trim(s.street_raw)), 'ё', 'е'),
                '(^|[[:space:]])(улица|ул|проспект|пр-кт|переулок|пер|шоссе|ш|набережная|наб|бульвар|б-р|проезд|площадь|пл|тракт|аллея|микрорайон|мкр)([.]|[[:space:]]|$)',
                ' '
            ),
            '[^[:alnum:]]+',
            ''
        ) as sphere_street,
        regexp_replace(
            replace(lower(trim(s.house_raw)), 'ё', 'е'),
            '[^[:alnum:]]+',
            ''
        ) as sphere_house,
        regexp_replace(
            replace(lower(trim(s.korpus_raw)), 'ё', 'е'),
            '[^[:alnum:]]+',
            ''
        ) as sphere_korpus,
        regexp_replace(
            replace(lower(trim(s.stroenie_raw)), 'ё', 'е'),
            '[^[:alnum:]]+',
            ''
        ) as sphere_stroenie,
        regexp_replace(
            replace(lower(trim(s.landmark_raw)), 'ё', 'е'),
            '[^[:alnum:]]+',
            ''
        ) as sphere_landmark
    from sphere_parts_raw s
),

sphere_core_keys as (
    /* Короткий список адресов ограничивает поиск по большой таблице ЕГРН. */
    select distinct
        sphere_locality,
        sphere_street,
        sphere_house
    from sphere_prepared
    where sphere_locality is not null
      and sphere_street is not null
      and sphere_house is not null
),

egrn_normalized as (
    /* Готовим адрес и минимальный набор данных ЕГРН. */
    select /*+ no_parallel(e) */
        coalesce(
            nullif(trim(e.cadaster), ''),
            'CAD_IND:' || to_char(e.cad_ind)
        ) as egrn_key,
        e.cad_ind,
        e.cadaster,
        e.egrn_address,
        e.square,
        case
            when regexp_like(
                replace(
                    regexp_replace(
                        trim(cast(e.square as varchar2(200))),
                        '[[:space:]]+',
                        ''
                    ),
                    ',',
                    '.'
                ),
                '^[0-9]+([.][0-9]+)?$'
            )
            then to_number(
                replace(
                    regexp_replace(
                        trim(cast(e.square as varchar2(200))),
                        '[[:space:]]+',
                        ''
                    ),
                    ',',
                    '.'
                ),
                '999999999999999999999999D9999999999',
                'NLS_NUMERIC_CHARACTERS=''.,'''
            )
        end as egrn_area,
        e.measure,
        e.building_type,
        e.oks_type,
        e.oks_purpose,
        e.object_status,
        e.fias_level,
        e.fias_id_house,
        e.row_update_date,
        e.ias_update_date,
        regexp_replace(trim(e.postal_code), '[^0-9]+', '')
            as egrn_postal_code,
        regexp_replace(
            regexp_replace(
                replace(lower(trim(e.region)), 'ё', 'е'),
                '(^|[[:space:]])(область|обл|край|республика|респ)([.]|[[:space:]]|$)',
                ' '
            ),
            '[^[:alnum:]]+',
            ''
        ) as egrn_region,
        regexp_replace(
            regexp_replace(
                replace(lower(trim(e.city)), 'ё', 'е'),
                '(^|[[:space:]])(город|г)([.]|[[:space:]]|$)',
                ' '
            ),
            '[^[:alnum:]]+',
            ''
        ) as egrn_city,
        regexp_replace(
            regexp_replace(
                replace(lower(trim(e.settlement)), 'ё', 'е'),
                '(^|[[:space:]])(пгт|поселок городского типа|рабочий поселок|р[.]?п|поселок|пос|село|с|деревня|д|хутор|х)([.]|[[:space:]]|$)',
                ' '
            ),
            '[^[:alnum:]]+',
            ''
        ) as egrn_settlement,
        regexp_replace(
            regexp_replace(
                replace(lower(trim(e.street)), 'ё', 'е'),
                '(^|[[:space:]])(улица|ул|проспект|пр-кт|переулок|пер|шоссе|ш|набережная|наб|бульвар|б-р|проезд|площадь|пл|тракт|аллея|микрорайон|мкр)([.]|[[:space:]]|$)',
                ' '
            ),
            '[^[:alnum:]]+',
            ''
        ) as egrn_street,
        regexp_replace(
            replace(lower(trim(e.house_number)), 'ё', 'е'),
            '[^[:alnum:]]+',
            ''
        ) as egrn_house,
        regexp_replace(
            replace(lower(trim(e.vladenie)), 'ё', 'е'),
            '[^[:alnum:]]+',
            ''
        ) as egrn_vladenie,
        regexp_replace(
            replace(lower(trim(e.korpus)), 'ё', 'е'),
            '[^[:alnum:]]+',
            ''
        ) as egrn_korpus,
        regexp_replace(
            replace(lower(trim(e.stroenie)), 'ё', 'е'),
            '[^[:alnum:]]+',
            ''
        ) as egrn_stroenie
    from DM_RISK_AVATAR.EGRN_DATA e
    where upper(trim(e.fias_level)) = 'FIAS_HOUSE'
      and lower(trim(e.oks_type)) in (
          'здание',
          'сооружение',
          'строение'
      )
      and e.flat is null
      and e.flat2 is null
      and e.office is null
      and e.office2 is null
      and e.room is null
      and e.room2 is null
      and e.compartment1 is null
      and e.compartment2 is null
      and (
          e.cadaster is not null
          or e.cad_ind is not null
      )
),

egrn_candidates as (
    /* Оставляем только адреса, которые могут относиться к нашей выгрузке. */
    select e.*
    from egrn_normalized e
    join sphere_core_keys k
        on k.sphere_street = e.egrn_street
       and k.sphere_house in (e.egrn_house, e.egrn_vladenie)
       and k.sphere_locality in (e.egrn_city, e.egrn_settlement)
),

address_matches as (
    /* Сравниваем адрес только до уровня здания. */
    select
        s.sphere_row_id,
        s.sphere_area,
        e.*
    from sphere_prepared s
    join egrn_candidates e
       on s.sphere_street = e.egrn_street
       and s.sphere_house in (e.egrn_house, e.egrn_vladenie)
       and s.sphere_locality in (e.egrn_city, e.egrn_settlement)
       and (
           s.sphere_region is null
           or e.egrn_region is null
           or s.sphere_region = e.egrn_region
       )
       and (
           s.sphere_postal_code is null
           or e.egrn_postal_code is null
           or s.sphere_postal_code = e.egrn_postal_code
       )
       and (
           s.sphere_korpus is null
           or s.sphere_korpus = e.egrn_korpus
       )
       and (
           s.sphere_stroenie is null
           or s.sphere_stroenie = e.egrn_stroenie
       )
    where s.sphere_locality is not null
      and s.sphere_street is not null
      and s.sphere_house is not null
),

ranked_egrn_rows as (
    /* Один кадастровый объект может повторяться. Оставляем свежую запись. */
    select
        m.*,
        row_number() over (
            partition by m.sphere_row_id, m.egrn_key
            order by
                m.row_update_date desc nulls last,
                m.ias_update_date desc nulls last,
                m.cad_ind desc nulls last
        ) as egrn_row_number
    from address_matches m
),

one_row_per_egrn_object as (
    select r.*
    from ranked_egrn_rows r
    where r.egrn_row_number = 1
),

area_check as (
    /* Площадь проверяем только там, где она есть с обеих сторон. */
    select
        c.*,
        count(*) over (
            partition by c.sphere_row_id
        ) as address_candidate_count,
        case
            when c.sphere_area > 0
             and c.egrn_area is not null
             and abs(c.egrn_area - c.sphere_area)
                 <= greatest(1, c.sphere_area * 0.01)
                then 1
            else 0
        end as area_matches
    from one_row_per_egrn_object c
),

area_choice as (
    select
        a.*,
        max(a.area_matches) over (
            partition by a.sphere_row_id
        ) as has_area_match
    from area_check a
),

candidates_after_area as (
    /* Если площадь помогла, оставляем совпавших. Иначе никого не отсекаем. */
    select a.*
    from area_choice a
    where a.has_area_match = 0
       or a.area_matches = 1
),

candidate_counts as (
    select
        c.*,
        count(*) over (
            partition by c.sphere_row_id
        ) as candidate_count
    from candidates_after_area c
),

candidate_summary as (
    select
        c.sphere_row_id,
        max(c.candidate_count) as candidate_count,
        max(c.address_candidate_count) as address_candidate_count,
        max(c.has_area_match) as has_area_match
    from candidate_counts c
    group by c.sphere_row_id
),

chosen_egrn as (
    /* Здание ЕГРН присоединяется только при одном кандидате. */
    select c.*
    from candidate_counts c
    where c.candidate_count = 1
)

select /*+ no_parallel */
    sphere.sphere_row_id as "sphere_row_id",

    /* Договор и основные ID строки Сферы. */
    sphere.contract_number as "Номер договора",
    sphere.contract_id as "ID договора",
    sphere.task_id as "ID задачи",
    sphere.task_object_link_id as "ID связи задачи и объекта",
    sphere.characteristics_id as "ID характеристик",
    sphere.object_id as "ID объекта Сферы",
    sphere.geo_address_id as "ID адреса Сферы",
    sphere.real_estate_objects_in_contract
        as "Объектов недвижимости в договоре",

    /* Объект и исходные адресные строки. */
    sphere.object_description as "Описание объекта Сферы",
    sphere.full_address as "Полный адрес Сферы",
    sphere.original_address as "Исходный адрес Сферы",
    prepared.source_address as "Адрес, который разбирал запрос",
    sphere.total_area as "Площадь Сферы",

    /* Части адреса, которые уже лежали в отдельных колонках Сферы. */
    sphere.postal_code as "Сфера: почтовый индекс",
    sphere.settlement as "Сфера: населённый пункт",
    sphere.street as "Сфера: улица",
    sphere.house as "Сфера: дом",
    sphere.block as "Сфера: корпус",
    sphere.building as "Сфера: строение",
    sphere.flat as "Сфера: квартира или помещение",
    sphere.office as "Сфера: офис",

    /* Так запрос разбил исходную строку адреса до очистки. */
    prepared.postal_code_raw as "После разбора: почтовый индекс",
    prepared.region_raw as "После разбора: регион",
    prepared.locality_raw as "После разбора: населённый пункт",
    prepared.street_raw as "После разбора: улица",
    prepared.house_raw as "После разбора: дом",
    prepared.korpus_raw as "После разбора: корпус",
    prepared.stroenie_raw as "После разбора: строение",
    prepared.landmark_raw as "После разбора: ориентир или территория",

    /* Итог поиска. */
    case
        when prepared.source_address is null
            then 'В Сфере нет адреса'
        when prepared.sphere_locality is null
          or prepared.sphere_street is null
          or prepared.sphere_house is null
            then 'Не удалось выделить населённый пункт, улицу или дом'
        when nvl(summary.candidate_count, 0) = 0
            then 'Здание ЕГРН не найдено'
        when summary.candidate_count = 1
            then 'Найдено одно здание ЕГРН'
        else 'Найдено несколько зданий. ЕГРН не присоединён'
    end as "Результат поиска",
    nvl(summary.address_candidate_count, 0)
        as "Кандидатов по адресу",
    nvl(summary.candidate_count, 0)
        as "Кандидатов после площади",
    case
        when summary.candidate_count = 1 then 1
        else 0
    end as "Соединение ЕГРН единичное",
    case
        when prepared.source_address is null
            then 'Нет адреса в Сфере'
        when prepared.sphere_locality is null
          or prepared.sphere_street is null
          or prepared.sphere_house is null
            then 'Адрес не удалось разобрать до здания'
        when nvl(summary.candidate_count, 0) = 0
            then 'Совпадение ЕГРН не найдено'
        when summary.candidate_count > 1
            then 'Неоднозначное совпадение'
        when summary.has_area_match = 1
            then 'Адрес здания и площадь'
        when prepared.sphere_area is null
            then 'Только адрес здания: в Сфере нет площади'
        when chosen.square is null
            then 'Только адрес здания: в ЕГРН нет площади'
        else 'Только адрес здания: площадь не подтвердила совпадение'
    end as "Основание соединения ЕГРН",
    case
        when summary.address_candidate_count > summary.candidate_count
            then 'Да'
        else 'Нет'
    end as "Площадь помогла сузить поиск",

    /* Очищенные значения показаны парами: Сфера и найденное здание КХД. */
    prepared.sphere_postal_code as "Сравнение: индекс Сферы",
    chosen.egrn_postal_code as "Сравнение: индекс КХД",

    prepared.sphere_region as "Сравнение: регион Сферы",
    chosen.egrn_region as "Сравнение: регион КХД",

    prepared.sphere_locality as "Сравнение: населённый пункт Сферы",
    case
        when prepared.sphere_locality = chosen.egrn_city
            then chosen.egrn_city
        when prepared.sphere_locality = chosen.egrn_settlement
            then chosen.egrn_settlement
    end as "Сравнение: населённый пункт КХД",

    prepared.sphere_street as "Сравнение: улица Сферы",
    chosen.egrn_street as "Сравнение: улица КХД",

    prepared.sphere_house as "Сравнение: дом Сферы",
    case
        when prepared.sphere_house = chosen.egrn_house
            then chosen.egrn_house
        when prepared.sphere_house = chosen.egrn_vladenie
            then chosen.egrn_vladenie
    end as "Сравнение: дом КХД",

    prepared.sphere_korpus as "Сравнение: корпус Сферы",
    chosen.egrn_korpus as "Сравнение: корпус КХД",

    prepared.sphere_stroenie as "Сравнение: строение Сферы",
    chosen.egrn_stroenie as "Сравнение: строение КХД",

    /* Поля одного найденного здания. */
    chosen.cad_ind as "Внутренний ID здания ЕГРН",
    chosen.cadaster as "Кадастровый номер здания",
    chosen.egrn_address as "Адрес здания ЕГРН",
    chosen.square as "Площадь здания ЕГРН",
    chosen.measure as "Единица площади здания",
    chosen.building_type as "Тип строения здания ЕГРН",
    chosen.oks_type as "Тип здания ЕГРН",
    chosen.oks_purpose as "Назначение здания ЕГРН",
    chosen.object_status as "Статус здания ЕГРН",
    chosen.fias_level as "Уровень адреса ЕГРН",
    chosen.fias_id_house as "ФИАС дома ЕГРН",
    case
        when chosen.cad_ind is not null then 'Здание'
    end as "Уровень присоединения"

from sphere_source sphere
join sphere_prepared prepared
    on prepared.sphere_row_id = sphere.sphere_row_id
left join candidate_summary summary
    on summary.sphere_row_id = sphere.sphere_row_id
left join chosen_egrn chosen
    on chosen.sphere_row_id = sphere.sphere_row_id
order by
    sphere.contract_number,
    sphere.object_id

/*
Как читать результат
--------------------
Найдено одно здание ЕГРН
    Здание присоединено ко всем объектам Сферы с этим адресом.

Найдено несколько зданий
    По адресу есть несколько кадастровых зданий. Ничего не присоединено.

Здание ЕГРН не найдено
    Адрес удалось разобрать, но запись уровня FIAS_HOUSE не найдена.

Не удалось выделить населённый пункт, улицу или дом
    Адрес есть, но его недостаточно для безопасного автоматического поиска.
*/

"""


In [14]:
# выполняем Oracle SQL и получаем одну строку результата на строку Сферы
if not KHD_DATA_SCHEMA.replace('_', '').isalnum():
    raise ValueError('Некорректное имя схемы КХД')

egrn_query = egrn_sql.replace(
    'DM_RISK_AVATAR.EGRN_DATA',
    f'{KHD_DATA_SCHEMA.upper()}.EGRN_DATA',
)


invalid_egrn_columns = [
    column for column in egrn_source_columns
    if not column.replace('_', '').isalnum()
]
if invalid_egrn_columns:
    raise ValueError(
        'В EGRN_DATA найдены некорректные имена колонок: '
        + ', '.join(invalid_egrn_columns)
    )

egrn_oracle_aliases = {
    column: f'ER{index:03d}'
    for index, column in enumerate(egrn_source_columns, start=1)
}
egrn_all_fields_sql = ',\n    '.join(
    f'chosen."{column}" as "{egrn_oracle_aliases[column]}"'
    for column in egrn_source_columns
)
egrn_query = egrn_query.replace(
    '    /* Поля одного найденного здания. */',
    '    ' + egrn_all_fields_sql
    + ',\n\n    /* Поля одного найденного здания. */',
)
with khd_connection.cursor() as cursor:
    sphere_json_bind = cursor.var(oracledb.DB_TYPE_CLOB)
    sphere_json_bind.setvalue(0, sphere_json)
    try:
        cursor.execute(egrn_query, sphere_json=sphere_json_bind)
    except oracledb.DatabaseError as error:
        oracle_error = error.args[0]
        error_offset = getattr(oracle_error, 'offset', None)
        if error_offset:
            fragment_start = max(0, error_offset - 300)
            fragment_end = min(len(egrn_query), error_offset + 300)
            print('Позиция ошибки Oracle:', error_offset)
            print(egrn_query[fragment_start:fragment_end])
        raise
    egrn_result_columns = [column[0] for column in cursor.description]
    egrn_result_rows = cursor.fetchall()

egrn_lookup_df = pd.DataFrame(
    egrn_result_rows,
    columns=egrn_result_columns,
)
egrn_raw_name_map = {
    oracle_alias: f'egrn_raw__{column.lower()}'
    for column, oracle_alias in egrn_oracle_aliases.items()
}
egrn_lookup_df = egrn_lookup_df.rename(
    columns={
        'SPHERE_ROW_ID': 'sphere_row_id',
        **egrn_raw_name_map,
    }
)

egrn_column_names = {
    'Результат поиска': 'egrn_match_status',
    'Кандидатов по адресу': 'egrn_address_candidate_count',
    'Кандидатов после площади': 'egrn_candidate_count',
    'Соединение ЕГРН единичное': 'egrn_is_unique_match',
    'Основание соединения ЕГРН': 'egrn_match_basis',
    'Площадь помогла сузить поиск': 'egrn_area_narrowed_search',
    'Адрес, который разбирал запрос': 'sphere_address_for_egrn',
    'После разбора: почтовый индекс': 'parsed_postal_code',
    'После разбора: регион': 'parsed_region',
    'После разбора: населённый пункт': 'parsed_locality',
    'После разбора: улица': 'parsed_street',
    'После разбора: дом': 'parsed_house',
    'После разбора: корпус': 'parsed_korpus',
    'После разбора: строение': 'parsed_stroenie',
    'После разбора: ориентир или территория': 'parsed_landmark',
    'Сравнение: индекс Сферы': 'compared_sphere_postal_code',
    'Сравнение: индекс КХД': 'matched_egrn_postal_code',
    'Сравнение: регион Сферы': 'compared_sphere_region',
    'Сравнение: регион КХД': 'matched_egrn_region',
    'Сравнение: населённый пункт Сферы': 'compared_sphere_locality',
    'Сравнение: населённый пункт КХД': 'matched_egrn_locality',
    'Сравнение: улица Сферы': 'compared_sphere_street',
    'Сравнение: улица КХД': 'matched_egrn_street',
    'Сравнение: дом Сферы': 'compared_sphere_house',
    'Сравнение: дом КХД': 'matched_egrn_house',
    'Сравнение: корпус Сферы': 'compared_sphere_korpus',
    'Сравнение: корпус КХД': 'matched_egrn_korpus',
    'Сравнение: строение Сферы': 'compared_sphere_stroenie',
    'Сравнение: строение КХД': 'matched_egrn_stroenie',
    'Внутренний ID здания ЕГРН': 'egrn_cad_ind',
    'Кадастровый номер здания': 'egrn_cadaster',
    'Адрес здания ЕГРН': 'egrn_address',
    'Площадь здания ЕГРН': 'egrn_square',
    'Единица площади здания': 'egrn_measure',
    'Тип строения здания ЕГРН': 'egrn_building_type',
    'Тип здания ЕГРН': 'egrn_oks_type',
    'Назначение здания ЕГРН': 'egrn_oks_purpose',
    'Статус здания ЕГРН': 'egrn_object_status',
    'Уровень адреса ЕГРН': 'egrn_fias_level',
    'ФИАС дома ЕГРН': 'egrn_fias_id_house',
    'Уровень присоединения': 'egrn_join_level',
}
egrn_lookup_df = egrn_lookup_df.rename(columns=egrn_column_names)
egrn_raw_columns = [
    f'egrn_raw__{column.lower()}'
    for column in egrn_source_columns
]
egrn_columns = (
    ['sphere_row_id']
    + list(egrn_column_names.values())
    + egrn_raw_columns
)

missing_egrn_columns = [
    column for column in egrn_columns
    if column not in egrn_lookup_df.columns
]
if missing_egrn_columns:
    raise ValueError(
        'Oracle не вернул ожидаемые колонки: '
        + ', '.join(missing_egrn_columns)
    )
if egrn_lookup_df['sphere_row_id'].duplicated().any():
    raise ValueError('Oracle вернул несколько строк для одного объекта Сферы')

expanded_egrn_df = sphere_with_row_id.merge(
    egrn_lookup_df[egrn_columns],
    on='sphere_row_id',
    how='left',
    validate='one_to_one',
)

if len(expanded_egrn_df) != len(sphere_with_row_id):
    raise ValueError('После соединения с ЕГРН изменилось количество строк')

print('Строк после соединения с ЕГРН:', len(expanded_egrn_df))


DatabaseError: ORA-00904: "CHOSEN"."UPDATE_DTTM": invalid identifier
Help: https://docs.oracle.com/error-help/db/ora-00904/

# 6. Проверка соединения с ЕГРН


In [ ]:
egrn_profile = (
    expanded_egrn_df['egrn_match_status']
    .fillna('Нет результата Oracle')
    .value_counts(dropna=False)
    .rename_axis('Результат поиска')
    .reset_index(name='Количество строк')
)
display(egrn_profile)


# 7. Статистический паспорт

Для анализа используется результат после соединения Сферы с ЕГРН. Каждая колонка результата попадёт в паспорт отдельной строкой.


In [ ]:
df = expanded_egrn_df.copy()
df.columns = [str(column).strip() for column in df.columns]

results_dir = OUTPUT_DIR
output_file = results_dir / 'паспорт_всех_признаков_расширенный.csv'
dictionary_file = (
    PROJECT_ROOT
    / 'МАТЕРИАЛЫ_ПРОЕКТА'
    / '00_входные_материалы'
    / 'колонки_таблиц.xlsx'
)

target_column = 'insured_sum'
csv_separator = ';'
csv_encoding = 'utf-8-sig'
top_values_limit = 10

if target_column not in df.columns:
    raise ValueError(f'В датасете нет целевой колонки {target_column}')

print('Строк для анализа:', len(df))
print('Колонок для анализа:', len(df.columns))
print('Итоговый паспорт:', output_file)


In [ ]:
# словарь нужен только для понятных названий и исходных таблиц
dictionary_rows = pd.DataFrame()

if dictionary_file.exists():
    dictionary_parts = []
    for sheet_name in ['Сфера', 'КХД 1.0']:
        part = pd.read_excel(dictionary_file, sheet_name=sheet_name)
        part['SOURCE_SYSTEM'] = sheet_name
        dictionary_parts.append(part)
    dictionary_rows = pd.concat(dictionary_parts, ignore_index=True)
    dictionary_rows.columns = [
        str(column).strip().upper()
        for column in dictionary_rows.columns
    ]
    dictionary_rows['TABLE_NAME_KEY'] = (
        dictionary_rows['TABLE_NAME'].astype('string').str.upper()
    )
    dictionary_rows['COLUMN_NAME_KEY'] = (
        dictionary_rows['COLUMN_NAME'].astype('string').str.lower()
    )

print('Строк в словаре:', len(dictionary_rows))


In [ ]:
technical_columns = {
    'sphere_row_id',
    'row_source',
    'contract_link_status',
    'contract_count',
    'has_contract',
    'has_address',
    'has_target',
    'target_status',
    'source_address',
    'input_address',
    'cdi_house_fias_candidate_count',
    'cdi_match_status',
    'cdi_is_unique_match',
    'egrn_address_candidate_count',
    'egrn_candidate_count',
    'egrn_is_unique_match',
    'egrn_match_method',
    'pipeline_match_status',
    'external_snapshot_at',
}

identifier_columns = {
    'contract_id',
    'contract_number',
    'previous_contract_id',
    'root_contract_id',
    'request_id',
    'task_id',
    'task_object_link_id',
    'characteristics_id',
    'object_id',
    'geo_address_id',
    'policyholder_id',
    'corporate_crm_id',
    'policyholder_inn',
    'policyholder_cdi_id',
    'policyholder_ogrn',
    'policyholder_kpp',
    'address_dgis_id',
    'cdi_fias_id_house',
    'egrn_cad_ind',
    'egrn_cadaster',
    'egrn_fias_id_house',
}

target_derived_columns = {
    'has_target',
    'target_status',
    'condition_min_insured_sum',
    'condition_max_insured_sum',
    'task_object_insured_sum',
    'contract_insured_sum',
}

sensitive_pattern = re.compile(
    r'(address|адрес|inn|ogrn|kpp|cadaster|contract_number|'
    r'policyholder_name|company_name|description|comment|json|'
    r'phone|email|(^|_)id($|_))',
    flags=re.IGNORECASE,
)

date_pattern = re.compile(
    r'(date|time|timestamp|d_create|d_change|d_delete|_start|_end|as_of)',
    flags=re.IGNORECASE,
)


def infer_source(column):
    if column in technical_columns:
        return 'Технический'
    if column.startswith('egrn_'):
        return 'ЕГРН'
    if column.startswith('cdi_'):
        return 'CDI'
    return 'Сфера'


def infer_role(column):
    if column == target_column:
        return 'target'
    if column == 'as_of_date':
        return 'split_key'
    if column in technical_columns:
        return 'service'
    if column in identifier_columns or column.endswith('_id'):
        return 'identifier'
    return 'feature'


In [ ]:
exact_table_map = {
    'insured_sum': 'BASE_INSURANCE_OBJECT_CONDITIONS',
    'full_address': 'BASE_GEO_ADDRESS',
    'original_address': 'BASE_INSURANCE_OBJECT',
    'object_description': 'BASE_INSURANCE_OBJECT',
    'total_area': 'BASE_INSURANCE_OBJECT_CHARACTERISTICS',
    'policyholder_inn': 'BPS_CONTRACTOR',
    'crm_industry': 'BPS_CORPORATE_CRM',
}


def dictionary_info(column, source):
    if dictionary_rows.empty or source in {'CDI', 'Технический'}:
        return None, None, None

    lookup_column = column
    table_filter = None

    if source == 'ЕГРН':
        lookup_column = column.removeprefix('egrn_')
        table_filter = 'EGRN_DATA'
    elif column in exact_table_map:
        table_filter = exact_table_map[column]

    matches = dictionary_rows.loc[
        dictionary_rows['COLUMN_NAME_KEY'].eq(lookup_column.lower())
    ].copy()

    if table_filter is not None:
        matches = matches.loc[
            matches['TABLE_NAME_KEY'].eq(table_filter)
        ]

    if matches.empty:
        return None, None, None

    row = matches.iloc[0]
    return (
        row.get('TABLE_NAME'),
        row.get('COLUMN_NAME'),
        row.get('COLUMN_COMMENTS'),
    )


# 3. Функции статистического анализа


In [ ]:
def clean_string_series(series):
    return (
        series.astype('string')
        .str.replace('\u00a0', ' ', regex=False)
        .str.strip()
        .replace('', pd.NA)
    )


def numeric_view(series):
    if pd.api.types.is_bool_dtype(series):
        return pd.Series(np.nan, index=series.index), 0.0
    if pd.api.types.is_numeric_dtype(series):
        converted = pd.to_numeric(series, errors='coerce')
    else:
        cleaned = (
            clean_string_series(series)
            .str.replace(' ', '', regex=False)
            .str.replace(',', '.', regex=False)
        )
        converted = pd.to_numeric(cleaned, errors='coerce')

    original_filled = clean_string_series(series).notna().sum()
    ratio = (
        converted.notna().sum() / original_filled
        if original_filled
        else 0.0
    )
    return converted, ratio


def datetime_view(series):
    converted = pd.to_datetime(series, errors='coerce')
    original_filled = clean_string_series(series).notna().sum()
    ratio = (
        converted.notna().sum() / original_filled
        if original_filled
        else 0.0
    )
    return converted, ratio


def infer_data_type(column, series, role):
    filled = series.dropna()
    if filled.empty:
        return 'unknown'
    if 'json' in column.lower():
        return 'json'
    if pd.api.types.is_bool_dtype(series):
        return 'boolean'

    text_values = set(
        clean_string_series(filled)
        .dropna()
        .str.lower()
        .unique()
        .tolist()
    )
    boolean_values = {
        'true', 'false', 't', 'f', 'yes', 'no', 'да', 'нет', '0', '1'
    }
    if text_values and text_values.issubset(boolean_values):
        return 'boolean'

    if date_pattern.search(column):
        _, date_ratio = datetime_view(series)
        if date_ratio >= 0.70:
            return 'datetime'

    if role != 'identifier':
        _, numeric_ratio = numeric_view(series)
        if numeric_ratio >= 0.95:
            return 'numeric'

    unique_count = filled.nunique(dropna=True)
    if unique_count <= 100 or unique_count / len(filled) <= 0.20:
        return 'category'
    return 'text'


In [ ]:
def source_available_mask(frame, source):
    if source == 'ЕГРН' and 'egrn_is_unique_match' in frame.columns:
        return pd.to_numeric(
            frame['egrn_is_unique_match'],
            errors='coerce',
        ).eq(1)
    if source == 'CDI' and 'cdi_is_unique_match' in frame.columns:
        return pd.to_numeric(
            frame['cdi_is_unique_match'],
            errors='coerce',
        ).eq(1)
    return pd.Series(True, index=frame.index)


def safe_top_values(series, feature, role, limit=10):
    if role == 'identifier' or sensitive_pattern.search(feature):
        return 'скрыто: конфиденциальное поле'

    clean = clean_string_series(series).fillna('NULL')
    counts = clean.value_counts(dropna=False).head(limit)
    total = len(clean)
    parts = []

    for value, count in counts.items():
        value_text = str(value).replace('\n', ' ').replace('|', '/')[:80]
        percent = count / total * 100 if total else 0
        parts.append(f'{value_text} [{count}, {percent:.2f}%]')

    return ' | '.join(parts)


def eta_squared(categories, target):
    pair = pd.DataFrame({'category': categories, 'target': target}).dropna()
    if len(pair) < 10 or pair['category'].nunique() < 2:
        return np.nan, len(pair)

    overall_mean = pair['target'].mean()
    total_variation = ((pair['target'] - overall_mean) ** 2).sum()
    if total_variation == 0:
        return np.nan, len(pair)

    grouped = pair.groupby('category')['target'].agg(['count', 'mean'])
    between_variation = (
        grouped['count'] * (grouped['mean'] - overall_mean) ** 2
    ).sum()
    return float(between_variation / total_variation), len(pair)


In [ ]:
def leakage_assessment(column, source, role):
    if role == 'target':
        return 'not_applicable', 'целевая переменная'
    if column in target_derived_columns:
        return 'high', 'поле напрямую связано с расчётом или наличием target'
    if role == 'identifier':
        return 'not_applicable', 'технический идентификатор'
    if role == 'service':
        return 'medium', 'служебное поле pipeline, не бизнес-признак'
    if source == 'ЕГРН' and any(
        word in column.lower()
        for word in ['update', 'actual', 'status', 'registration_date']
    ):
        return 'high', 'нужно проверить, было ли значение доступно на дату договора'
    if source in {'ЕГРН', 'CDI'}:
        return 'unknown', 'нужно подтвердить исторический срез внешнего источника'
    if date_pattern.search(column):
        return 'unknown', 'нужно проверить доступность на as_of_date'
    return 'unknown', 'требуется бизнес-проверка доступности на дату расчёта'


def preliminary_decision(role, filled_pct, unique_count, quality_flags, leakage):
    if role == 'target':
        return 'target', 'целевая переменная'
    if role == 'identifier':
        return 'exclude', 'идентификатор оставляем только для связи и контроля'
    if role == 'service':
        return 'exclude', 'служебное поле не подаём в модель'
    if 'all_missing' in quality_flags:
        return 'exclude', 'колонка полностью пустая'
    if 'constant' in quality_flags:
        return 'exclude', 'в колонке одно заполненное значение'
    if leakage == 'high':
        return 'check', 'возможна утечка данных'
    if filled_pct < 5:
        return 'check', 'заполнено меньше 5% строк'
    return 'check', 'решение принимается после бизнес-проверки и baseline-модели'


# 4. Общие метрики датасета


In [ ]:
target_numeric, target_numeric_ratio = numeric_view(df[target_column])
if target_numeric_ratio < 0.95:
    raise ValueError(
        f'Колонка {target_column} не распознана как числовая'
    )

dataset_metrics = {
    'rows': len(df),
    'columns': len(df.columns),
    'target_column': target_column,
    'target_filled': int(target_numeric.notna().sum()),
    'target_missing': int(target_numeric.isna().sum()),
    'target_zero': int(target_numeric.eq(0).sum()),
    'target_positive': int(target_numeric.gt(0).sum()),
}

if 'object_id' in df.columns:
    dataset_metrics['unique_objects'] = int(df['object_id'].nunique(dropna=True))
if 'contract_id' in df.columns:
    dataset_metrics['unique_contracts'] = int(
        df['contract_id'].nunique(dropna=True)
    )
if {'task_id', 'object_id'}.issubset(df.columns):
    duplicate_mask = df.duplicated(['task_id', 'object_id'], keep=False)
    dataset_metrics['duplicate_task_object_rows'] = int(duplicate_mask.sum())
if 'full_address' in df.columns:
    dataset_metrics['full_address_filled'] = int(
        clean_string_series(df['full_address']).notna().sum()
    )
if 'cdi_is_unique_match' in df.columns:
    dataset_metrics['cdi_unique_matches'] = int(
        pd.to_numeric(df['cdi_is_unique_match'], errors='coerce').eq(1).sum()
    )
if 'egrn_is_unique_match' in df.columns:
    dataset_metrics['egrn_unique_matches'] = int(
        pd.to_numeric(df['egrn_is_unique_match'], errors='coerce').eq(1).sum()
    )
if 'cdi_match_status' in df.columns:
    dataset_metrics['cdi_ambiguous_matches'] = int(
        df['cdi_match_status'].eq('ambiguous_house_fias').sum()
    )
if 'egrn_match_method' in df.columns:
    dataset_metrics['egrn_ambiguous_matches'] = int(
        df['egrn_match_method'].eq('ambiguous').sum()
    )
if 'as_of_date' in df.columns:
    as_of = pd.to_datetime(df['as_of_date'], errors='coerce')
    dataset_metrics['as_of_date_min'] = (
        as_of.min().date().isoformat() if as_of.notna().any() else None
    )
    dataset_metrics['as_of_date_max'] = (
        as_of.max().date().isoformat() if as_of.notna().any() else None
    )

display(
    pd.DataFrame(
        dataset_metrics.items(),
        columns=['Показатель', 'Значение'],
    )
)


# 5. Паспорт всех признаков


In [ ]:
feature_rows = []

for number, feature in enumerate(df.columns, start=1):
    raw_series = df[feature]
    if pd.api.types.is_object_dtype(raw_series) or pd.api.types.is_string_dtype(raw_series):
        series = clean_string_series(raw_series)
    else:
        series = raw_series
    source = infer_source(feature)
    role = infer_role(feature)
    data_type = infer_data_type(feature, series, role)
    available_mask = source_available_mask(df, source)

    total_rows = len(series)
    source_available_rows = int(available_mask.sum())
    filled_count = int(series.notna().sum())
    missing_count = int(series.isna().sum())
    filled_pct = filled_count / total_rows * 100 if total_rows else 0.0
    filled_among_available = int(series.loc[available_mask].notna().sum())
    filled_among_available_pct = (
        filled_among_available / source_available_rows * 100
        if source_available_rows
        else np.nan
    )
    unique_count = int(series.nunique(dropna=True))
    unique_pct = unique_count / filled_count * 100 if filled_count else 0.0

    row = {
        'record_type': 'feature',
        'metric_name': None,
        'metric_value': None,
        'feature': feature,
        'russian_name': feature.replace('_', ' '),
        'source': source,
        'source_table': None,
        'source_column': None,
        'source_comment': None,
        'data_type': data_type,
        'role': role,
        'total_rows': total_rows,
        'source_available_rows': source_available_rows,
        'filled_count': filled_count,
        'missing_count': missing_count,
        'filled_pct': round(filled_pct, 4),
        'filled_among_available_pct': (
            round(filled_among_available_pct, 4)
            if not pd.isna(filled_among_available_pct)
            else np.nan
        ),
        'unique_count': unique_count,
        'unique_pct': round(unique_pct, 4),
        'zero_count': None,
        'negative_count': None,
        'outlier_iqr_count': None,
        'min': None,
        'p01': None,
        'p25': None,
        'median': None,
        'mean': None,
        'p75': None,
        'p99': None,
        'max': None,
        'top_values': None,
        'target_relation_method': None,
        'target_relation_value': None,
        'target_relation_rows': None,
        'quality_flag': None,
        'available_at_prediction_time': (
            'no' if role == 'target'
            else 'not_applicable' if role in {'identifier', 'service'}
            else 'unknown'
        ),
        'leakage_risk': None,
        'leakage_reason': None,
        'preliminary_decision': None,
        'decision_reason': None,
    }

    source_table, source_column, source_comment = dictionary_info(feature, source)
    row['source_table'] = source_table
    row['source_column'] = source_column
    row['source_comment'] = source_comment
    if source_comment is not None and not pd.isna(source_comment):
        row['russian_name'] = str(source_comment)

    quality_flags = []
    if filled_count == 0:
        quality_flags.append('all_missing')
    elif unique_count == 1:
        quality_flags.append('constant')
    if 0 < filled_pct < 5:
        quality_flags.append('coverage_lt_5pct')
    elif 5 <= filled_pct < 20:
        quality_flags.append('coverage_lt_20pct')
    if filled_count and unique_pct > 95 and role == 'feature':
        quality_flags.append('high_cardinality')

    if data_type == 'numeric':
        numeric, _ = numeric_view(series)
        valid = numeric.dropna()
        if not valid.empty:
            quantiles = valid.quantile([0.01, 0.25, 0.50, 0.75, 0.99])
            q1 = quantiles.loc[0.25]
            q3 = quantiles.loc[0.75]
            iqr = q3 - q1
            if iqr > 0:
                outlier_count = int(
                    ((valid < q1 - 1.5 * iqr) | (valid > q3 + 1.5 * iqr)).sum()
                )
            else:
                outlier_count = 0

            row.update({
                'zero_count': int(valid.eq(0).sum()),
                'negative_count': int(valid.lt(0).sum()),
                'outlier_iqr_count': outlier_count,
                'min': valid.min(),
                'p01': quantiles.loc[0.01],
                'p25': q1,
                'median': quantiles.loc[0.50],
                'mean': valid.mean(),
                'p75': q3,
                'p99': quantiles.loc[0.99],
                'max': valid.max(),
            })
            if row['negative_count']:
                quality_flags.append('contains_negative')
            if row['zero_count']:
                quality_flags.append('contains_zero')
            if outlier_count:
                quality_flags.append('iqr_outliers')

            pair = pd.DataFrame({
                'feature': numeric,
                'target': target_numeric,
            }).dropna()
            if feature != target_column and len(pair) >= 10:
                row['target_relation_method'] = 'spearman'
                row['target_relation_value'] = pair['feature'].corr(
                    pair['target'],
                    method='spearman',
                )
                row['target_relation_rows'] = len(pair)

    elif data_type == 'datetime':
        dates, _ = datetime_view(series)
        valid_dates = dates.dropna()
        if not valid_dates.empty:
            row['min'] = valid_dates.min().isoformat()
            row['max'] = valid_dates.max().isoformat()
            pair = pd.DataFrame({
                'feature': dates.map(
                    lambda value: value.toordinal() if pd.notna(value) else np.nan
                ),
                'target': target_numeric,
            }).dropna()
            if feature != target_column and len(pair) >= 10:
                row['target_relation_method'] = 'spearman_date'
                row['target_relation_value'] = pair['feature'].corr(
                    pair['target'],
                    method='spearman',
                )
                row['target_relation_rows'] = len(pair)

    elif data_type in {'category', 'boolean'}:
        row['top_values'] = safe_top_values(
            series,
            feature,
            role,
            top_values_limit,
        )
        if role == 'feature' and 2 <= unique_count <= 50:
            eta, relation_rows = eta_squared(series, target_numeric)
            row['target_relation_method'] = 'eta_squared'
            row['target_relation_value'] = eta
            row['target_relation_rows'] = relation_rows

    elif data_type in {'text', 'json'}:
        row['top_values'] = (
            'скрыто: текстовое или конфиденциальное поле'
        )

    leakage_risk, leakage_reason = leakage_assessment(
        feature,
        source,
        role,
    )
    row['leakage_risk'] = leakage_risk
    row['leakage_reason'] = leakage_reason
    row['quality_flag'] = ' | '.join(quality_flags) if quality_flags else 'ok'

    decision, decision_reason = preliminary_decision(
        role,
        filled_pct,
        unique_count,
        quality_flags,
        leakage_risk,
    )
    row['preliminary_decision'] = decision
    row['decision_reason'] = decision_reason
    feature_rows.append(row)

    if number % 50 == 0:
        print('Обработано признаков:', number)

feature_passport = pd.DataFrame(feature_rows)
print('Признаков в паспорте:', len(feature_passport))


# 6. Добавление общих метрик в тот же CSV


In [ ]:
passport_columns = feature_passport.columns.tolist()
metric_rows = []

for metric_name, metric_value in dataset_metrics.items():
    row = {column: None for column in passport_columns}
    row['record_type'] = 'dataset_metric'
    row['metric_name'] = metric_name
    row['metric_value'] = metric_value
    metric_rows.append(row)

metric_passport = pd.DataFrame(metric_rows, columns=passport_columns)
passport = pd.concat(
    [metric_passport, feature_passport],
    ignore_index=True,
)

if len(feature_passport) != len(df.columns):
    raise ValueError('В паспорт попали не все колонки датасета')

if feature_passport['feature'].duplicated().any():
    raise ValueError('В паспорте появились повторяющиеся признаки')

display(
    feature_passport[[
        'feature',
        'source',
        'data_type',
        'role',
        'filled_pct',
        'unique_count',
        'quality_flag',
        'preliminary_decision',
    ]].head(30)
)


# 7. Контроль перед сохранением

Здесь выводятся только агрегаты. Реальные значения адресов, ИНН и идентификаторов не показываются.


In [ ]:
control = pd.DataFrame({
    'Показатель': [
        'Колонок во входном датасете',
        'Строк feature в паспорте',
        'Строк dataset_metric',
        'Всего строк итогового CSV',
        'Числовых признаков',
        'Категориальных признаков',
        'Идентификаторов',
        'Служебных полей',
        'Полностью пустых колонок',
        'Константных колонок',
        'Признаков с высоким риском утечки',
    ],
    'Значение': [
        len(df.columns),
        len(feature_passport),
        len(metric_passport),
        len(passport),
        feature_passport['data_type'].eq('numeric').sum(),
        feature_passport['data_type'].isin(['category', 'boolean']).sum(),
        feature_passport['role'].eq('identifier').sum(),
        feature_passport['role'].eq('service').sum(),
        feature_passport['quality_flag'].str.contains('all_missing').sum(),
        feature_passport['quality_flag'].str.contains('constant').sum(),
        feature_passport['leakage_risk'].eq('high').sum(),
    ],
})
display(control)


# 8. Сохранение одного CSV


In [ ]:
results_dir.mkdir(parents=True, exist_ok=True)
passport.to_csv(
    output_file,
    index=False,
    sep=csv_separator,
    encoding=csv_encoding,
)

print('Файл сохранён:', output_file)
print('Строк:', len(passport))
print('Признаков:', len(feature_passport))


In [ ]:
engine.dispose()
khd_connection.close()
print('Подключения к Сфере и КХД закрыты')
